# Phase 4 Memory Comparison

Colab notebook to compare the memory strategies using an already trained classifier and display the best memory method.

In [ ]:
!pip install tensorflow pandas numpy scikit-learn networkx tqdm -q

In [ ]:
import os
import sys

repo_path = '/content/AI_Agentic_DL'
if not os.path.exists(repo_path):
    !git clone https://github.com/Lawapaul/AI_Agentic_DL.git {repo_path}

os.chdir(repo_path)
!git -C {repo_path} fetch origin
!git -C {repo_path} checkout codex/retraining-loop-system
!git -C {repo_path} pull origin codex/retraining-loop-system

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print('Repo ready:', repo_path)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
processed_path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'

model_candidates = [
    os.path.join(repo_path, 'saved_models', 'hybrid_cnn_lstm_full_2_2m.keras'),
    os.path.join(repo_path, 'saved_models', 'hybrid_cnn_lstm_500k.keras'),
]
model_path = next((path for path in model_candidates if os.path.exists(path)), None)
if model_path is None:
    raise FileNotFoundError('No pretrained model found under saved_models/.')

output_csv = os.path.join(repo_path, 'experiments', 'results', 'memory_comparison_results.csv')
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

print('Processed data:', processed_path)
print('Model path:', model_path)
print('Output CSV:', output_csv)

In [ ]:
!python experiments/phase4_memory_pipeline.py \
  --processed_path "{processed_path}" \
  --model_path "{model_path}" \
  --compare_all \
  --batch_size 256 \
  --top_k 5 \
  --output "{output_csv}"

In [ ]:
import pandas as pd

results_df = pd.read_csv(output_csv)
results_df = results_df.sort_values('top1_retrieval_accuracy', ascending=False).reset_index(drop=True)
results_df

In [ ]:
best_row = results_df.iloc[0]
print('Best memory strategy:', best_row['strategy'])
print('Top-1 accuracy:', round(float(best_row['top1_retrieval_accuracy']), 4))
print('Top-5 accuracy:', round(float(best_row['top5_retrieval_accuracy']), 4))
print('Class purity:', round(float(best_row['class_purity']), 4))
print('Average cosine similarity:', round(float(best_row['avg_cosine_similarity']), 4))